# Traffic: PCMCI versus Ensemble

Este notebook executa a implementação atual em duas configurações sobre o mesmo recorte do dataset **CausalTime Traffic**:

1. **PCMCI isolado**;
2. **Ensemble robusto exatamente com a configuração padrão (`quick_mode=False`) do notebook principal**, usando todos os métodos registrados e selecionando combinações de dois ou três métodos.

Os dois resultados são comparados ao `graph.npy` com métricas de esqueleto não direcionado. Isso é necessário porque o grafo de referência é simétrico e não informa direção nem lag. Portanto, direção e atraso estimados pelos métodos são mostrados nas tabelas, mas não entram como acerto ou erro na avaliação estrutural.

## 1. Configuração reproduzível

Usamos exatamente o perfil `causaltime_traffic` ativo no notebook principal: trajetória 0, o mesmo subgrafo de seis nós, `max_lag=2` e `panel_max_lag=1`. O subgrafo contém 14 conexões verdadeiras em 15 pares possíveis; por ser muito denso, compare o F1 também com o baseline que prevê todos os pares.

In [ ]:
from pathlib import Path
import math
import os

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

from causal_discovery import (
    CausalPreprocessor,
    compute_ranked_undirected_skeleton_metrics,
    compute_undirected_skeleton_metrics,
    get_registered_method_kwargs,
    get_registered_method_weights,
    get_registered_methods,
    load_time_series_dataset,
    run_pcmci_multiple_trajectories,
)
from causal_discovery.ensemble_selection import select_robust_ensemble_combination

dataset_config = {
    "data_format": "causaltime",
    "data_path": Path("datasets/causaltime/traffic/gen_data.npy"),
    "graph_path": Path("datasets/causaltime/traffic/graph.npy"),
    "trajectory_index": 0,
    "column_prefix": "traffic",
    "selected_columns": [
        "traffic_02", "traffic_04", "traffic_06",
        "traffic_08", "traffic_13", "traffic_19",
    ],
    "decomposition_period": None,
    "max_lag": 2,
    "panel_max_lag": 1,
}

# Mesmos valores iniciais da execução padrão do notebook principal.
quick_mode = False
expert_knowledge = []
dataset_validation_threshold = 0.5


## 2. Carregamento e pré-processamento

A mesma trajetória e os mesmos dados processados são fornecidos aos dois experimentos. As 480 trajetórias são independentes e não são concatenadas, pois isso criaria transições temporais artificiais.

In [ ]:
dataset_bundle = load_time_series_dataset(
    dataset_config["data_path"],
    data_format=dataset_config["data_format"],
    selected_columns=dataset_config.get("selected_columns"),
    graph_path=dataset_config.get("graph_path"),
    trajectory_index=dataset_config.get("trajectory_index", 0),
    column_prefix=dataset_config.get("column_prefix", "variable"),
)
raw_data = dataset_bundle.data.copy()
data = raw_data.copy()
dataset_ground_truth = dataset_bundle.ground_truth.copy()
SELECTED_COLUMNS = list(dataset_bundle.selected_columns)

preprocessor = CausalPreprocessor(
    data,
    significance_level=0.05,
    decomposition_period=dataset_config.get("decomposition_period"),
)
processed_data = preprocessor.fit_transform(
    make_stationary=True,
    normalize=True,
    remove_trend=False,
    max_diffs=2,
)

max_lag = int(dataset_config.get("max_lag", 5))
preprocessing_summary = preprocessor.summary()

print("Shape da trajetória selecionada:", raw_data.shape)
print("Trajetórias disponíveis:", dataset_bundle.trajectory_count)
print("Nós avaliados:", SELECTED_COLUMNS)
print("Arestas direcionadas no graph.npy após o recorte:", len(dataset_ground_truth))
print("Observação: pares simétricos serão colapsados na avaliação.")
display(processed_data.head())
display(dataset_ground_truth.head(20))


## 3. Resultado apenas com PCMCI

O PCMCI já retorna apenas ligações direcionadas defasadas aceitas no nível de significância configurado. Para a comparação com o grafo, múltiplas direções/lags do mesmo par contam como uma única adjacência.

In [ ]:
all_candidate_methods = get_registered_methods()
all_candidate_method_kwargs = get_registered_method_kwargs(max_lag)
all_method_weights = get_registered_method_weights()

# Mesma função e os mesmos kwargs registrados usados pelo pipeline principal.
pcmci_result = all_candidate_methods["PCMCI"](
    processed_data, **all_candidate_method_kwargs["PCMCI"]
)
pcmci_result = pcmci_result.loc[
    pcmci_result["source"].ne(pcmci_result["target"])
].reset_index(drop=True)

print(f"PCMCI encontrou {len(pcmci_result)} arestas direcionadas com lag.")
display(pcmci_result.sort_values(["p_value", "source", "target"]).reset_index(drop=True))


## 4. Resultado com Ensemble robusto

A seleção usa todos os métodos retornados por `get_registered_methods()`, exatamente como a execução padrão do notebook principal (`quick_mode=False`): combinações de dois e três métodos, oito bootstraps, até quatro jobs e limite de 900 segundos. O `graph.npy` não participa da seleção. Os resultados-base e de bootstrap são pré-computados e reutilizados.

In [ ]:
def runtime_method_names(quick_mode):
    return list(all_candidate_methods)


def build_selection_args(quick_mode, n_bootstrap=None, parallel_jobs=None):
    quick_mode = bool(quick_mode)
    default_bootstraps = 4 if quick_mode else 8
    default_jobs = max(1, min(4, (os.cpu_count() or 2) - 1))
    return {
        "min_methods": 2,
        "max_methods": 2 if quick_mode else 3,
        "min_votes": 1,
        "n_bootstrap": int(n_bootstrap or default_bootstraps),
        "block_size": max(2, len(processed_data) // 12),
        "stability_threshold": 0.6,
        "selection_probability_threshold": 0.55,
        "prior_edge_probability": 0.1,
        "posterior_weight": 0.7,
        "adaptive_method_weights": True,
        "stability_weight": 0.65,
        "local_expert_weight": 0.60,
        "method_stability_power": 1.0,
        "method_diversity_bonus": 0.15,
        "method_density_penalty": 0.5,
        "minimum_method_weight": 0.05,
        "confidence_level": 0.95,
        "random_state": 42,
        "precompute_runs": True,
        "parallel_jobs": int(parallel_jobs or default_jobs),
        "max_bootstrap_seconds": 240 if quick_mode else 900,
    }


def restrict_method_relations(method, allowed_relations):
    allowed_relations = set(allowed_relations)

    def run_restricted(data, **kwargs):
        result = method(data, **kwargs)
        if result is None or result.empty:
            return result
        selected_rows = [
            (source, target) in allowed_relations
            for source, target in zip(result["source"], result["target"])
        ]
        return result.loc[selected_rows].reset_index(drop=True)

    return run_restricted


search_method_names = runtime_method_names(quick_mode)
common_selection_args = build_selection_args(quick_mode)
selected_relations = [
    (source, target)
    for source in processed_data.columns
    for target in processed_data.columns
    if source != target
]
selected_relation_set = set(selected_relations)
selected_methods = {
    name: restrict_method_relations(method, selected_relation_set)
    for name, method in all_candidate_methods.items()
}

num_methods = len(search_method_names)
combination_count = sum(
    math.comb(num_methods, size)
    for size in range(
        common_selection_args["min_methods"],
        min(common_selection_args["max_methods"], num_methods) + 1,
    )
)
print("Métodos candidatos:", search_method_names)
print("Combinações avaliadas:", combination_count)
print("Execuções pré-computadas:", num_methods * (common_selection_args["n_bootstrap"] + 1))
print("Parâmetros:", common_selection_args)

selection = select_robust_ensemble_combination(
    processed_data,
    selected_methods,
    method_kwargs=all_candidate_method_kwargs,
    method_weights=all_method_weights,
    expert_knowledge=list(expert_knowledge),
    **common_selection_args,
)

ensemble_result = selection["best_evaluation"]["probabilistic_summary"].copy()
ensemble_result = ensemble_result.loc[
    ensemble_result["source"].ne(ensemble_result["target"])
].reset_index(drop=True)
ensemble_detected = ensemble_result.loc[
    ensemble_result["edge_probability"] >= dataset_validation_threshold
].reset_index(drop=True)

print("Melhor combinação:", selection["best_combination"])
print("Pesos adaptativos:", selection["best_evaluation"]["effective_method_weights"])
display(selection["best_evaluation"]["method_weight_diagnostics"])
display(selection["ranking"])
print(f"Ensemble manteve {len(ensemble_detected)} arestas acima do limiar.")
ensemble_columns = [
    "source", "target", "lag", "votes", "edge_probability",
    "ensemble_score", "local_expert_score", "consensus_score",
    "dominant_method", "dominant_edge_stability",
    "confidence", "combined_p_value", "method",
]
display(ensemble_detected[ensemble_columns].sort_values(
    "edge_probability", ascending=False
).reset_index(drop=True))


## 5. Comparação com o grafo de referência

A avaliação abaixo ignora direção e lag porque essas informações não existem no `graph.npy`. Para o PCMCI, toda aresta retornada é considerada detectada. Para o ensemble, aplicamos exatamente o limiar `dataset_validation_threshold = 0.5` do notebook principal.

In [ ]:
pcmci_metrics = compute_undirected_skeleton_metrics(
    pcmci_result,
    dataset_ground_truth,
    nodes=SELECTED_COLUMNS,
    evaluated_relations=selected_relations,
)
ensemble_metrics = compute_undirected_skeleton_metrics(
    ensemble_result,
    dataset_ground_truth,
    prob_threshold=dataset_validation_threshold,
    nodes=SELECTED_COLUMNS,
    evaluated_relations=selected_relations,
)

metric_names = [
    "true_positives", "false_positives", "false_negatives",
    "precision", "recall", "f1_score",
    "structural_hamming_distance", "candidate_pairs",
    "ground_truth_pairs", "ground_truth_prevalence",
    "all_pairs_baseline_f1",
]
comparison = pd.DataFrame(
    {
        "PCMCI": {name: pcmci_metrics[name] for name in metric_names},
        "ENSEMBLE": {name: ensemble_metrics[name] for name in metric_names},
    }
).T
comparison["f1_minus_baseline"] = (
    comparison["f1_score"] - comparison["all_pairs_baseline_f1"]
)
display(comparison.round(3))

diagnostics = pd.DataFrame({
    "PCMCI": {
        "Pares corretos (TP)": pcmci_metrics["true_positive_pairs"],
        "Pares extras (FP)": pcmci_metrics["false_positive_pairs"],
        "Pares não recuperados (FN)": pcmci_metrics["false_negative_pairs"],
    },
    "ENSEMBLE": {
        "Pares corretos (TP)": ensemble_metrics["true_positive_pairs"],
        "Pares extras (FP)": ensemble_metrics["false_positive_pairs"],
        "Pares não recuperados (FN)": ensemble_metrics["false_negative_pairs"],
    },
})
display(diagnostics)


## 6. Evidência complementar PCMCI com todas as trajetórias

Esta é a mesma validação complementar do notebook principal. Ela preserva as fronteiras das 480 trajetórias, usa os 20 nós como contexto e restringe a apresentação aos pares avaliados. Os escores não alteram o PCMCI isolado nem o ensemble.

In [ ]:
panel_pair_scores = None
panel_ranking_metrics = None
if dataset_bundle.trajectory_count > 1:
    panel_max_lag = int(dataset_config.get("panel_max_lag", 1))
    panel_pair_scores = run_pcmci_multiple_trajectories(
        dataset_bundle.observed_trajectories(),
        dataset_bundle.available_columns,
        max_lag=panel_max_lag,
        standardize=True,
    )
    evaluated_pairs = {
        tuple(sorted((source, target)))
        for source, target in selected_relations
        if source != target
    }
    panel_pair_scores = panel_pair_scores.loc[[
        tuple(sorted((source, target))) in evaluated_pairs
        for source, target in zip(
            panel_pair_scores["source"], panel_pair_scores["target"]
        )
    ]].reset_index(drop=True)
    try:
        panel_ranking_metrics = compute_ranked_undirected_skeleton_metrics(
            panel_pair_scores, dataset_ground_truth
        )
        print(f"AUROC: {panel_ranking_metrics['roc_auc']:.3f}")
        print(f"Average precision: {panel_ranking_metrics['average_precision']:.3f}")
        print(f"Baseline aleatório de AP: {panel_ranking_metrics['random_average_precision']:.3f}")
    except ValueError as error:
        print(f"Ranking não calculado: {error}")
    display(panel_pair_scores.sort_values("score", ascending=False).head(20))


In [ ]:
def undirected_pairs(frame):
    if frame.empty:
        return set()
    return {
        tuple(sorted((str(source), str(target))))
        for source, target in zip(frame["source"], frame["target"])
        if source != target
    }

def adjacency_matrix(frame, ordered_nodes):
    pairs = undirected_pairs(frame)
    matrix = np.zeros((len(ordered_nodes), len(ordered_nodes)), dtype=int)
    for row, source in enumerate(ordered_nodes):
        for column, target in enumerate(ordered_nodes):
            if source != target and tuple(sorted((source, target))) in pairs:
                matrix[row, column] = 1
    return matrix

truth_matrix = adjacency_matrix(dataset_ground_truth, SELECTED_COLUMNS)
pcmci_matrix = adjacency_matrix(pcmci_result, SELECTED_COLUMNS)
ensemble_matrix = adjacency_matrix(ensemble_detected, SELECTED_COLUMNS)

figure = make_subplots(
    rows=1, cols=3,
    subplot_titles=("graph.npy (referência)", "PCMCI", "ENSEMBLE"),
    horizontal_spacing=0.08,
)
for column, matrix in enumerate(
    [truth_matrix, pcmci_matrix, ensemble_matrix], start=1
):
    figure.add_trace(
        go.Heatmap(
            z=matrix, x=SELECTED_COLUMNS, y=SELECTED_COLUMNS, zmin=0, zmax=1,
            colorscale=[[0, "#f1f5f9"], [1, "#2563eb"]],
            showscale=False, hovertemplate="origem=%{y}<br>destino=%{x}<br>aresta=%{z}<extra></extra>",
        ),
        row=1, col=column,
    )
figure.update_xaxes(tickangle=45)
figure.update_yaxes(autorange="reversed")
figure.update_layout(
    title="Esqueletos não direcionados: referência versus métodos",
    height=520, width=1250, margin=dict(t=90, b=130),
)
figure.show()


## 7. Como interpretar

- **Precision** mede quanto das adjacências estimadas pertence ao grafo de referência.
- **Recall** mede quanto do grafo de referência foi recuperado.
- **F1** equilibra precision e recall.
- **SHD** aqui é `FP + FN`; menor é melhor.
- **F1 menos baseline** é especialmente importante no subgrafo padrão, que é quase completo.

Diferenças pequenas nesta execução devem ser interpretadas com cautela: cada trajetória tem somente 40 passos, o recorte padrão é denso e o grafo do benchmark representa adjacências sem direção/lag. O resultado sugere evidência estrutural relativa; não constitui prova de causalidade no trânsito real. Para preservar a equivalência com o notebook principal, altere os parâmetros primeiro no perfil `causaltime_traffic` do principal e replique a mesma alteração neste notebook.